In [25]:
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path
import re
import time
import os
load_dotenv()  # looks for .env in the current/parent dirs

True

In [26]:
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
)

In [ ]:
def _extract_csv(text: str) -> str:
    """
    Return raw CSV content. Strips code fences like ```csv ... ``` if present.
    """
    if not text:
        return ""
    fence = re.search(r"```(?:csv)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        return fence.group(1).strip()
    return text.strip()

def build_report(report_txt_file: str, csv_override: bool = False, *, model: str = "gpt-5-nano") -> Path:
    """
    Given a report text filename like 'K123456.txt' (found in data/txt),
    create 'data/csv/K123456.csv' by asking the model to convert
    salient molecular markers to a CSV.
    """
    txt_path = Path("data/txt") / report_txt_file
    if txt_path.suffix.lower() != ".txt":
        raise ValueError(f"Expected a .txt file, got: {txt_path}")

    if not txt_path.exists():
        raise FileNotFoundError(f"Text report not found: {txt_path.resolve()}")

    csv_path = Path("data/csv") / (txt_path.stem + ".csv")
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    if csv_path.exists() and not csv_override:
        print(f"[build_report] CSV already exists, skipping (set csv_override=True to regenerate): {csv_path}")
        return csv_path

    report_text = txt_path.read_text(encoding="utf-8")

    system_prompt = (
        """
        You are given a histopathology / molecular pathology report for a GBM biopsy.  
        Your ONLY task is to output a CSV.  

        The CSV must always contain exactly 6 rows:  
        1 header row + 5 rows for the following markers, in this exact order:  
        - IDH1 or IDH2 mutated  
        - MGMT promoter methylation status  
        - EGFR amplification  
        - ATRX mutation  
        - Subtype  

        Formatting rules:  
        - Header row must be: marker,value,notes  
        - Marker column: use exactly the names shown above, in this exact order.  
        - Value column:  
        • IDH1 or IDH2 mutated → [yes/no/Not Reported]  
        • MGMT promoter methylation status → [methylated/unmethylated/Not Reported]  
        • EGFR amplification → [yes/no/Not Reported]  
        • ATRX mutation → [yes/no/Not Reported]  
        • Subtype → [classical/mesenchymal/proneural/neural/other/Not Reported]  
        - Notes column: leave blank unless the wording is unclear. If unclear, copy the exact ambiguous wording.  

        Hard rules:  
        1. Output must be valid RFC4180 CSV only (no prose, no markdown, no commentary).  
        2. The CSV must contain exactly 6 rows (header + 5). Never output more or fewer.  
        3. Never add any markers other than the 5 listed above.  
        4. If the report does not mention a marker, set its value to "Not Reported".  
        5. Do not reorder or rename markers.  

        Example of required shape (values are illustrative only):  

        marker,value,notes  
        IDH1 or IDH2 mutated,yes,  
        MGMT promoter methylation status,methylated,  
        EGFR amplification,Not Reported,  
        ATRX mutation,no,  
        Subtype,proneural,  
        """
    )

    user_prompt = (
        "Convert the following report to CSV per the instructions.\n"
        f"{report_text}\n"
    )

    # Small, simple retry in case of transient network/model issues
    last_err = None
    for attempt in range(3):
        try:
            response = client.responses.create(
                model=model,
                temperature=0.0,
                instructions=system_prompt,
                input=[
                    # {"role": "developer", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                service_tier="flex",
            )
            csv_text = _extract_csv(getattr(response, "output_text", "").strip())
            if not csv_text:
                raise RuntimeError("Model returned empty output_text or failed to produce CSV.")
            # Ensure CSV has the required header
            if not csv_text.splitlines()[0].strip().lower().startswith("marker,value"):
                csv_text = "marker,value\n" + csv_text.lstrip()
            csv_path.write_text(csv_text, encoding="utf-8", newline="\n")
            print(f"[build_report] Wrote CSV: {csv_path}")
            return csv_path
        except Exception as e:
            last_err = e
            wait = 1.5 * (attempt + 1)
            print(f"[build_report] Attempt {attempt+1}/3 failed: {e} — retrying in {wait:.1f}s")
            time.sleep(wait)

    # If we reached here, all attempts failed
    raise RuntimeError(f"Failed to build CSV for {txt_path.name}") from last_err

In [29]:
# fetch all .txt files in data/txt and process them
if __name__ == "__main__":
    txt_dir = Path("data/txt")
    if not txt_dir.exists():
        raise FileNotFoundError(f"Text directory not found: {txt_dir.resolve()}")

    txt_files = sorted(f.name for f in txt_dir.glob("*.txt"))
    print(f"Found {len(txt_files)} text reports to process.")
    print(txt_files)
    for txt_file in txt_files:
        try:
            build_report(txt_file, csv_override=False, model="gpt-5")
        except Exception as e:
            print(f"Error processing {txt_file}: {e}")

Found 9 text reports to process.
['K0284435.txt', 'K0501346.txt', 'K0611148.txt', 'K1038156.txt', 'K1279654.txt', 'K1463292.txt', 'K1618041.txt', 'K1900709.txt', 'K3646667.txt']
[build_report] Wrote CSV: data/csv/K0284435.csv


KeyboardInterrupt: 